# Spanish tutor: the data pipeline, step by step

This notebook walks through how the tutor's data is loaded, processed, vectorized and
stored, and why each design decision was made. It runs the project's own code
(`src/spanish_tutor/`), so what you see here is exactly what the tutor uses.

```
SUBTLEX-ESP ─┐                                  ┌─> word bank (SQLite)
Tatoeba ─────┼─> parse ─> lemmatize ─> filter ──┤
Wiktionary ──┘           (lemma, pos)           └─> embed ─> vector store (Chroma)
```

| Section | What it covers |
|---|---|
| 1. Data sets | The three sources, with links, licenses and raw samples |
| 2. Processing | How text becomes `(lemma, pos)` pairs, word frequencies, and the word bank |
| 3. Vectorization | The embedding model, one sentence's vector, and a 2D PCA check |
| 4. Storage | The SQL database (structure, constraints, trigger, example queries) and the Chroma vector store |
| 5. Lookups | Searching the store against the word bank, looking up words in SQLite, resolving a learner's own words, and finding readable examples |

**Before running:** the pipeline commands in the README must have been run, so that
`data/raw/`, `data/processed/word_bank.db` and `data/processed/chroma/` exist. Use the
project's `.venv` as the kernel.

**Licensing note:** the outputs below include excerpts of SUBTLEX-ESP (CC BY-NC-SA),
Tatoeba (CC BY 2.0 FR) and Wiktionary (CC BY-SA), plus your personal word bank. Clear
outputs before committing this notebook to the public repo.

In [ ]:
import itertools
import json
import warnings

# sentence-transformers imports tqdm, which warns that notebook progress widgets
# aren't installed. Harmless: we don't use progress bars here.
warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
from sklearn.decomposition import PCA

from spanish_tutor import db
from spanish_tutor.config import CHROMA_DIR, DB_PATH
from spanish_tutor.ingest import subtlex
from spanish_tutor.ingest.download import RAW_DIR, WIKTIONARY_FILE
from spanish_tutor.ingest.tatoeba import ANALYZED_PATH, read_analyzed, read_tsv
from spanish_tutor.ingest.wiktionary import Wiktionary

pd.set_option("display.max_colwidth", 90)
print("raw data:      ", RAW_DIR)
print("word bank:     ", DB_PATH)
print("vector store:  ", CHROMA_DIR)

## 1. Data sets

Three open data sources feed the pipeline. Each answers a different question:

| Source | Question it answers | License |
|---|---|---|
| SUBTLEX-ESP | How common is each word in everyday speech? | CC BY-NC-SA 4.0 |
| Tatoeba | How is each word actually used, in real sentences with translations? | CC BY 2.0 FR |
| Wiktionary | Is this a real word, and what does it mean? | CC BY-SA 4.0 |

**Design decision: the repository ships code, never data.** Each source is downloaded
at build time into `data/raw/` (gitignored), because the licenses attach conditions to
redistribution (non-commercial, share-alike, per-author credit).

### 1.1 SUBTLEX-ESP: word frequencies from film subtitles

- **What:** frequency counts for 94,261 Spanish word forms, from about 40 million words
  of film and TV subtitles.
- **Why subtitles:** subtitles approximate everyday spoken language better than books or
  news. The SUBTLEX authors found that subtitle frequencies predict how quickly people
  recognize words better than older written-text frequency lists did.
- **Source:** Cuetos, F., Glez-Nosti, M., Barbón, A., & Brysbaert, M. (2011).
  *SUBTLEX-ESP: Spanish word frequencies based on film subtitles.* Psicológica, 32, 133–143.
  Project page: [osf.io/xp6sz](https://osf.io/xp6sz/), file: [SUBTLEX-ESP.xlsx](https://osf.io/download/fxt57/).

The raw file has an unusual layout: one sheet with the list split into **three
side-by-side column blocks**, sorted alphabetically. Here are its first rows:

In [ ]:
workbook = openpyxl.load_workbook(RAW_DIR / "SUBTLEX-ESP.xlsx", read_only=True)
raw_rows = list(itertools.islice(workbook.worksheets[0].iter_rows(values_only=True), 6))
workbook.close()
pd.DataFrame(
    raw_rows[1:], columns=[f"{c} ({i // 5 + 1})" if c else "" for i, c in enumerate(raw_rows[0])]
)

`subtlex.load_counts()` reads all three blocks, normalizes each form (Unicode NFC,
lowercase) and sums the 76 words that appear twice. The result maps each **word form**
to its count. It has no lemmas and no parts of speech: *hablo*, *hablas* and *habló* are
separate rows. Section 2 deals with that.

In [ ]:
form_counts = subtlex.load_counts(RAW_DIR / "SUBTLEX-ESP.xlsx")
total = sum(form_counts.values())
print(f"{len(form_counts):,} word forms, {total:,} subtitle tokens\n")

top = pd.DataFrame(form_counts.most_common(15), columns=["form", "count"])
top["per million"] = (top["count"] / total * 1e6).round(0).astype(int)
top

**Design decision: accents are kept.** Spanish uses accents to tell words apart, so
normalization never strips them:

In [ ]:
pd.DataFrame(
    [(f, form_counts[f]) for f in ["si", "sí", "esta", "está", "el", "él", "que", "qué"]],
    columns=["form", "count"],
)

### 1.2 Tatoeba: real sentences with translations

- **What:** a community-built collection of sentences and their translations, made for
  language learners. It has 442,260 Spanish sentences, 261,405 of which have an English
  translation.
- **Why:** it supplies real usage. That serves three purposes: deciding how ambiguous
  forms are used, providing example sentences, and grounding retrieval.
- **Source:** [tatoeba.org/en/downloads](https://tatoeba.org/en/downloads). We use the
  weekly per-language exports:
  - [spa_sentences_detailed.tsv.bz2](https://downloads.tatoeba.org/exports/per_language/spa/spa_sentences_detailed.tsv.bz2) (id, language, text, author, dates)
  - [eng_sentences.tsv.bz2](https://downloads.tatoeba.org/exports/per_language/eng/eng_sentences.tsv.bz2) (id, language, text)
  - [spa-eng_links.tsv.bz2](https://downloads.tatoeba.org/exports/per_language/spa/spa-eng_links.tsv.bz2) (Spanish id → English id)
- **License:** CC BY 2.0 FR. Each sentence's author must be credited, so the author
  travels with every sentence through the whole pipeline.

The raw files are tab-separated with no header row, and use `\N` for "unknown":

In [ ]:
for name in ["spa_sentences_detailed.tsv.bz2", "eng_sentences.tsv.bz2", "spa-eng_links.tsv.bz2"]:
    print(f"--- {name}")
    for row in itertools.islice(read_tsv(RAW_DIR / name), 3):
        print("   ", row)

`tatoeba.load_sentences()` joins the three files: each Spanish sentence gets its
lowest-id English translation (if any) and its author. A sample of the result:

In [ ]:
sample = [s for s in itertools.islice(read_analyzed(ANALYZED_PATH), 2000, 2400) if s.en][:8]
pd.DataFrame(
    [(s.id, s.es, s.en, s.author) for s in sample], columns=["id", "Spanish", "English", "author"]
)

### 1.3 Wiktionary: is it a real word, and what does it mean?

- **What:** English Wiktionary's entries for Spanish words, in machine-readable form.
- **Source:** [kaikki.org](https://kaikki.org/dictionary/Spanish/), which extracts
  Wiktionary with *wiktextract* (Ylonen, 2022, LREC). We use the
  [Spanish-only extract](https://kaikki.org/dictionary/Spanish/kaikki.org-dictionary-Spanish.jsonl)
  (~95 MB compressed in transit) and fall back to the full multi-language dump if it
  disappears.
- **License:** CC BY-SA 4.0.

`ingest/download.py` reduces each entry to its word, part of speech and up to five
senses. **Design decision:** Wiktionary lists inflected forms as their own entries
(*pies* = "plural of *pie*"). Those are dropped for nouns, verbs and adjectives, so only
dictionary forms remain. They are *kept* for function words, because a learner has to
learn *ti* ("prepositional of *tú*") as a separate word.

In [ ]:
wikt_path = RAW_DIR / WIKTIONARY_FILE
entries = {}
with wikt_path.open(encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        if entry["word"] in {"bajo", "ti"}:
            entries.setdefault(entry["word"], []).append(entry)

for word, word_entries in entries.items():
    for entry in word_entries[:3]:
        print(f"{word} ({entry['pos']})")
        for sense in entry["senses"][:3]:
            print(f"    - {sense['gloss']}   tags={sense['tags']}")

The `Wiktionary` class indexes those entries by `(lemma, UD part of speech)`, the same
key the word bank uses. Two design decisions matter here:

- **Part-of-speech fallback for function words.** spaCy and Wiktionary sometimes disagree:
  spaCy tags *cómo* as a pronoun, Wiktionary calls it an adverb. For function words only,
  a compatible part of speech counts as a match. The word keeps spaCy's tag, since that's
  what content indexing will produce.
- **Following "alternative spelling" entries.** *ese* is listed only as "alternative
  spelling of *ése*", so the lookup follows it to the real definition.

In [ ]:
wiktionary = Wiktionary(wikt_path)
pd.DataFrame(
    [
        (lemma, pos, wiktionary.definition(lemma, pos))
        for lemma, pos in [
            ("bajo", "ADJ"),
            ("bajo", "ADP"),
            ("cómo", "PRON"),
            ("ese", "PRON"),
            ("ti", "PRON"),
        ]
    ],
    columns=["lemma", "pos", "definition"],
)

## 2. Processing: from text to `(lemma, pos)`

The word bank stores **words**, not word forms. If you know *hablar*, you can read
*hablamos* and *habló*. So all text, whether in subtitles, sentences or (later) songs and
books, is reduced to `(lemma, part of speech)` pairs.

**Design decision: a word is identified by its lemma *and* part of speech.** *bajo* the
adjective ("short"), *bajo* the preposition ("under") and *bajo* the noun ("bass") are
different words to learn, so they're tracked separately. The cost is that tagging
mistakes create spurious pairs. We accepted that cost.

**Design decision: one lemmatizer for everything.** `src/spanish_tutor/lexicon.py` is the
only code that turns text into `(lemma, pos)`. The word bank, the seed pipeline, the
vector store and the future content index all use it. If two parts of the system
lemmatized differently, they would disagree about whether a word is new.

### 2.1 What spaCy produces

We use [spaCy's `es_dep_news_trf`](https://spacy.io/models/es#es_dep_news_trf): a
transformer pipeline built on BETO (a Spanish BERT), trained on the AnCora Spanish
treebank. The transformer predicts each token's part of speech and morphology (tense,
person, number, ...) from its whole sentence; a rule-based lemmatizer then uses those to
find the dictionary form. Here is its raw output for one sentence:

In [ ]:
from spanish_tutor.lexicon import analyze, load_nlp, token_analyses, vocabulary

sentence = "Me lo dijo ella: no vengo del mercado porque hay mucho trabajo."
doc = load_nlp()(sentence)
raw = pd.DataFrame(
    [(t.text, t.lemma_, t.pos_, str(t.morph)) for t in doc],
    columns=["token", "spaCy lemma", "spaCy POS", "morphology"],
)
raw

#### Design decision: a transformer tagger (2026-10-02)

The project started with the smaller `es_core_news_md` (word vectors plus a CNN). The first
real conversation exposed its weak spot: in *y riego las plantas* ("and I water the
plants"), *riego* was tagged as the noun "irrigation" and taught as a new word, although
the learner had just been taught *regar*. Spanish has many such noun/verb pairs (*trabajo*,
*juego*, *uso*, *canto*), and learners write in the first person all the time.

Measured on hand-labeled Tatoeba samples of tokens `es_core_news_md` had tagged NOUN but
that are also verb forms:

| | Verbs found (of 51) | Real nouns wrongly made verbs (of 52) | Learner-style sentences (of 8) |
|---|---|---|---|
| `es_core_news_md` | 0 | 0 | 3 |
| `es_core_news_lg` | 11 | 0 | 4 |
| `es_dep_news_trf` | **50** | **0** | **6** |

Across the whole corpus the switch changed 2.0% of the 3.2 million tokens. In a hand check
of 50 random changes, the transformer was right in 39 and the old model in 5 (5 ties, 1
wrong in both). It also recovers 14,600 tokens the old model dropped, mostly
sentence-initial verbs it mistook for names (*Lee el artículo*, *Llueve*, *Sígueme*), and
resolves *ven* as *venir* rather than *ver*.

The cost: about 10 ms per sentence instead of under 1 ms (90 minutes for all of Tatoeba
instead of 15), a 6-second model load, and a pinned `transformers` version. A
conversation turn takes several seconds anyway, so the tagger's time doesn't show.

The same homographs, tagged by the current pipeline:

In [ ]:
homographs = [
    "Yo trabajo en una oficina.",
    "El trabajo es duro.",
    "Por la mañana riego las plantas.",
    "El riego de las plantas es importante.",
    "Cuando compras algo, tienes que pagarlo.",
    "Hice las compras ayer.",
]
rows = [
    (text, form, ", ".join(f"{l}|{p}" for l, p in analyses))
    for text, tokens in zip(homographs, analyze(homographs))
    for form, analyses in tokens
    if form in {"trabajo", "riego", "compras"}
]
pd.DataFrame(rows, columns=["sentence", "token", "analysis"])

Several of those raw results are wrong *for our purpose*, so `lexicon.normalize()`
applies these rules:

| Rule | Example | Why |
|---|---|---|
| Unicode NFC + lowercase, **accents kept** | *Sí* → *sí* (≠ *si*) | Accents distinguish words |
| `AUX` → `VERB` | *hay* (haber, AUX) → haber VERB | spaCy tags copular *ser/estar* and auxiliary *haber* as AUX; without folding, each verb would be two words |
| Contractions expanded | *del* → *de* ADP + *el* DET | spaCy leaves *del*/*al* whole |
| **Personal pronouns keep their own form** | *me* → *me* (not *yo*) | spaCy lemmatizes *me, nos, conmigo* to *yo* and *se, lo, le, ella* to *él*. Knowing *yo* doesn't mean knowing *conmigo* |
| Clitic verbs reduced to the verb | *dar yo él* → *dar* | spaCy joins attached pronouns into the lemma |
| Non-vocabulary dropped | names (capitalized), punctuation, digits | Not words to learn |
| **Lowercase "names" re-tagged** | *perro* PROPN → *perro* NOUN | See below |
| **Missing accents restored**, before tagging | *detras* → *detrás*, *manana* → *mañana* | See below |
| **Lemma correction against Wiktionary** | *has* → *haber*, *deberiar* → *deber* | See below |

The same sentence after normalization. Changed rows are marked:

In [ ]:
from spanish_tutor.lexicon import load_corrector

corrector = load_corrector()
normalized = [
    ", ".join(f"{l}|{p}" for l, p in token_analyses(t, corrector)) or "(dropped)" for t in doc
]
raw["normalized (lemma|POS)"] = normalized
raw["changed"] = ["✓" if n != f"{t.lemma_}|{t.pos_}" else "" for t, n in zip(doc, normalized)]
raw[["token", "spaCy lemma", "spaCy POS", "normalized (lemma|POS)", "changed"]]

#### Lemma correction

spaCy's lemmatizer sometimes produces a lemma that isn't a word: it leaves an inflected
form unchanged (*has*, *dólares*) or invents one (*deberías* → *deberiar*). Every such
mistake would become a fake "word" in the lexicon, and a known word would look new.

**Design decision: repair lemmas with Wiktionary's own form records.** Wiktionary lists
which dictionary word each inflected form belongs to (*has* → *haber*). `LemmaCorrector`
applies these rules when spaCy's `(lemma, pos)` isn't a dictionary word:

1. If the surface form itself is a dictionary word with that POS, use it (*oír*).
2. If the form is a listed form of a dictionary word with that POS, use that word
   (*has* → *haber*). Several candidates are decided by SUBTLEX frequency:
   *crees* is a form of both *crear* and *creer*, and *creer* is far more common.
3. If the surface form is a dictionary word under exactly one *other* POS, the tag was
   wrong (*en serio* tagged NOUN → *serio* ADJ).
4. Otherwise a listed form under another POS. Ties are never guessed.
5. **Misspellings that Wiktionary lists as entries are respelled.** Wiktionary has
   entries for common typos ("*dia*: misspelling of *día*"), so they pass the dictionary
   check, and Tatoeba contains them: 35 had become lexicon words of their own. An entry
   whose every sense points to the same word, at least one tagged as a misspelling, is
   replaced by that word and corrected in turn (*habia* → *había* → *haber*).
   "Alternative forms" are left alone: *buen* (of *bueno*) and *mi* (of *mío*) are
   separate words a learner has to learn. The rule also helps with learner typing:
   *tambien*, *aser* (for *hacer*), *haci* (for *así*).

6. **A lemma that is a real word, but not the form's word, is replaced.** Both spaCy
   models lemmatize the verb *riego* ("I water") as *regir*, a real verb that *riego* isn't
   a form of; Wiktionary lists it under *regar*. When the form is listed under other words
   with the same POS and not under the tagger's lemma, the most common of those wins
   (*verte* → *ver*, not *verter*; *buena* → *bueno*, not the separate word *buen*;
   *cuántas* → *cuánto*). Measured on Tatoeba (2026-10-02), it applies to verbs,
   adjectives and determiners: for pronouns Wiktionary files *eso* under the old spelling
   *ése*, so all 13k pronoun hits would have been wrong, and noun hits were mostly
   spelling variants. For verbs only infinitives count (the participle entry *hecho*
   doesn't replace *hacer*), and a pronominal entry (*quejarse*) doesn't split its verb.
   A hand check of 50 hits on the full corpus: 46 right; the misses were rare regional
   or archaic forms (*podes*, *plega*, *retiñe*).

7. **A few measured single-word fixes** (`LEMMA_FIXES`), only where a general rule
   isn't possible. One so far: the transformer files the thanks *gracias* under *gracia*
   ("grace") in 451 of 1,127 Tatoeba tokens, so the tutor's "¡Gracias!" taught *gracia*
   as a new word. The plural of *gracia* in that sense is vanishingly rare, so *gracias*
   is never filed under it (the tagger's part of speech is kept). It applies even without
   the corrector, so it doesn't show in the table below.

Irregular comparatives (*peor*, *mejor*, *mayor*, *menor*) stay separate words: Wiktionary
calls them forms of *malo*, *bueno*, ..., but a learner has to learn them on their own.

Raw spaCy (`corrector=None`) vs. corrected, on sentences with typical mistakes:

In [ ]:
import copy

examples = [
    "¿Crees que deberías ir?",
    "Has visto los dólares.",
    "Lo dije en serio.",
    "Déjame en paz, vámonos.",
    "Tambien fue un buen dia.",
    "Por la mañana riego las plantas.",
    "Es una buena idea.",
    "¿Cuántas veces quieres verte?",
]
# Expressions (en serio, dejar en paz) have their own section below; leave them out here.
words_only = copy.copy(corrector)
words_only.expressions = None
rows = []
for text, raw_tokens, fixed_tokens in zip(
    examples, analyze(examples, corrector=None), analyze(examples, corrector=words_only)
):
    for (form, before), (_, after) in zip(raw_tokens, fixed_tokens):
        if before != after:
            rows.append(
                (
                    text,
                    form,
                    ", ".join(f"{l}|{p}" for l, p in before),
                    ", ".join(f"{l}|{p}" for l, p in after),
                )
            )
pd.DataFrame(rows, columns=["sentence", "token", "spaCy", "corrected"])

Every Tatoeba run writes all corrections to `data/processed/lemma_corrections.csv` for
review. The most frequent ones:

In [ ]:
corrections = pd.read_csv(DB_PATH.parent / "lemma_corrections.csv", encoding="utf-8-sig")
fixed = corrections[corrections["lemma"].notna()]
unresolved = corrections[corrections["lemma"].isna()]
print(
    f"{fixed.tokens.sum():,} tokens corrected ({len(fixed):,} distinct cases); "
    f"{unresolved.tokens.sum():,} tokens left unresolved"
)
fixed.head(12)

#### Common words mistaken for names

spaCy sometimes tags an ordinary word as a proper noun (`PROPN`), even in lowercase in
the middle of a sentence. Names aren't vocabulary, so until 2026-10-01 these words were
silently dropped: about 10,000 Tatoeba tokens of some 1,650 dictionary words, led by *tu*,
*perro*, *llover* and *madre*. A dropped word is invisible everywhere downstream. It is
missing from the lexicon's frequencies and from sentence difficulty, and the conversation
skill can't notice it in a reply, so it never gets taught.

**Design decision: a lowercase `PROPN` that the dictionary knows is a tagging error.**
Names are capitalized, so only tokens written in lowercase are re-tagged, by
`LemmaCorrector.retag_proper_noun`:

1. A personal pronoun form is a pronoun (*contigo*).
2. A function-word reading beats a noun sense: Wiktionary's noun *mi* is the musical
   note E, not the possessive.
3. A headword with one part of speech gets it (*madre*, *llover*).
4. A headword with several, including NOUN, becomes a NOUN, since the tagger saw it in a
   noun slot (*perro*, *rosa*).
5. Anything else is treated as an inflected form, like any other correction
   (*quieres* → *querer*).

Ties (*tuyo*: pronoun or determiner?) are reported, never guessed. Words the dictionary
doesn't know at all (*toki pona*, English words) stay dropped. The rule runs inside
`normalize()`, so every consumer gets it without an extra step: Tatoeba ingestion, the
lexicon, the vector index, and live text.

Rules 1 and 2 came from checking a first version against the word bank: 2.7% of
re-tagged tokens had landed on a part of speech the learner didn't have, mostly *mi* as a
noun and *contigo* as an adverb. Each would have been flagged and taught as a new word.

The transformer tagger adopted on 2026-10-02 (below) makes this mistake far less often: the
re-tag now fires on about 1,500 Tatoeba tokens instead of 10,000. It still happens, mostly
with institutions (*gobierno*, *embajada*, *senado*). In this sentence, *Juan* stays a name
and *gobierno* is repaired:

In [ ]:
examples = ["Juan dice que el gobierno miente."]
rows = [
    (
        text,
        token.text,
        token.pos_,
        ", ".join(f"{l}|{p}" for l, p in token_analyses(token, corrector)) or "(dropped: a name)",
    )
    for text, parsed in zip(examples, load_nlp().pipe(examples))
    for token in parsed
    if token.pos_ == "PROPN"
]
pd.DataFrame(rows, columns=["sentence", "token", "spaCy POS", "after re-tagging"])

Across the whole corpus, from the corrections file:

In [ ]:
propn = corrections[corrections["tagger_pos"] == "PROPN"]
retagged = propn[propn["lemma"].notna()]
print(
    f"{retagged.tokens.sum():,} tokens re-tagged ({len(retagged):,} distinct words); "
    f"{propn[propn['lemma'].isna()].tokens.sum():,} tokens left as ties"
)
retagged.head(10)

#### Words typed without their accents

On an English keyboard, learners leave accents out, and some Tatoeba authors do too. In
the first real conversation, *detras* and *manana* never reached the word bank, for two
different reasons:

- spaCy lemmatized *detras* as the plural of an invented noun *detra*, so no repair of the
  lemma could recover *detrás*.
- Accent matching kept *ñ* distinct from *n* on purpose (*año* vs. *ano*), so *manana*
  couldn't match *mañana*.

**Design decision: restore the accents in the text, before tagging.** Then spaCy sees
*detrás* in its sentence and tags it correctly. `AccentRestorer` replaces a word only when
all of these hold:

1. It was typed without any marks. Typed accents are never changed, so *Sudán* is left
   alone.
2. It isn't a word as typed. *esta*, *ano* and *hable* are real words, so they stay. A
   spelling that is only a misspelling or obsolete entry (*jardin*) doesn't count as a
   word.
3. Ignoring accents and *ñ*, exactly one word or inflected form matches, or one clearly
   more common in the subtitles (*niño*, not the rare *ñiño*). Ties are never guessed.
4. That word occurs at least 20 times in SUBTLEX-ESP. Rarer restorations were mostly junk:
   *ana* → *aña*, *canadá* → *cañada*.
5. If capitalized, it starts a sentence (*Despues, …*). Elsewhere a capital marks a name:
   *País de Gales* must not become *galés*.

The first version had neither rule 1 nor rule 5. The review file showed *sudán* → *sudan*
and *Gales* → *Galés*, which is why both rules exist.

The learner's words from that first conversation, and two names that must not change:

In [ ]:
examples = [
    "Dos al frente y tres detras de mi casa.",
    "Me gusta trabajar en el jardin por la manana.",
    "Despues, rastrillo las hojas.",
    "Vivo en el País de Gales.",
    "Viajo a Sudán con Maria.",
]
pd.DataFrame(
    [(text, corrector.restorer.restore_text(text)) for text in examples],
    columns=["typed", "restored before tagging"],
)

Across Tatoeba, every restoration is written to a review file:

In [ ]:
restorations = pd.read_csv(DB_PATH.parent / "accent_restorations.csv", encoding="utf-8-sig")
print(f"{restorations.tokens.sum():,} tokens restored ({len(restorations):,} distinct spellings)")
restorations.head(12)

#### Multi-word expressions

Some words only mean something together: *sin embargo* is "however", and its *embargo*
is not the "seizure" a dictionary gives for the word alone. In a real conversation the
tutor taught *embargo* for exactly this reason. So fixed expressions are one vocabulary
item, matched by lemma sequence: inflection is free (*me di cuenta* is *darse cuenta*),
and a contraction counts with both its parts (*al menos*).

**Design decision: an approved list, not every dictionary phrase.** Wiktionary has about
15,600 multi-word Spanish entries, but matching all of them is harmful. The most frequent
matches were function-word pairs (*de la*, *a la*, *lo que*), phrasebook sentences whose
words you know separately (*no sé*, *dónde estás*), and wrong senses: *la vida* is listed
only as slang for prostitution. `ingest/expressions.py` builds the list in three steps:

1. **Rule filters:** drop phrasebook, article, proverb and name entries, and senses that
   are literal, vulgar, archaic, obsolete or just spellings; keep phrases found in at
   least 3 Tatoeba sentences. 3,066 candidates.
2. **Review:** Claude (Opus 5.5, Batch API) sees each candidate's senses and real Tatoeba
   sentences, and decides whether it's a fixed expression whose listed meaning is the
   usual one there, and which sense. The reviews are kept, with the reviewer. A phrase
   kept with no fitting sense is dropped rather than taught with a wrong definition:
   *a punto* is listed only as "at the ready", but Tatoeba uses it in *a punto de*
   ("about to"), which is approved as its own expression.
3. **Matching** with the approved phrases (`lexicon.ExpressionMatcher`): the first word
   carries the expression, the other words carry nothing, so the expression is credited
   and taught, never its parts.

The review's decisions on some of the most frequent candidates:

In [ ]:
from spanish_tutor.ingest.expressions import (
    CANDIDATES_PATH,
    EXPRESSIONS_PATH,
    REVIEWS_PATH,
    read_jsonl,
)

candidates = {c["phrase"]: c for c in read_jsonl(CANDIDATES_PATH)}
reviews = read_jsonl(REVIEWS_PATH)
kept = sum(1 for r in reviews if r.get("keep"))
approved = len(read_jsonl(EXPRESSIONS_PATH))
print(f"{len(candidates):,} candidates reviewed by {reviews[0]['reviewer']}; {kept:,} kept")
print(f"{approved:,} approved: kept, with a listed sense that fits how the corpus uses them")
frequent = sorted(reviews, key=lambda r: -candidates[r["phrase"]]["sentences"])[:16]
pd.DataFrame(
    [
        (
            r["phrase"],
            candidates[r["phrase"]]["sentences"],
            "keep" if r["keep"] else "reject",
            r["reason"],
        )
        for r in frequent
    ],
    columns=["phrase", "sentences", "decision", "reason"],
)

In a sentence, each expression becomes one item:

In [ ]:
sentence_with_expressions = "Sin embargo, a veces me doy cuenta de que tengo razón."
pd.DataFrame(
    [
        (form, ", ".join(f"{l}|{p}" for l, p in analyses) or "(part of the expression)")
        for form, analyses in next(analyze([sentence_with_expressions]))
    ],
    columns=["token", "analysis"],
)

`analyze()` runs all of this over many texts at once (using spaCy's batched `nlp.pipe`),
and `vocabulary()` reduces a sentence to its set of distinct words:

In [ ]:
tokens = next(analyze([sentence]))
print("per token:", tokens[:6], "...")
print("\nvocabulary:", sorted(vocabulary(tokens)))

Lemmatizing all 442k Tatoeba sentences takes about 90 minutes on CPU, so
`python -m spanish_tutor.ingest.tatoeba` does it once and caches the result in
`data/processed/tatoeba_analyzed.jsonl`. The tagger that made it is recorded in
`tatoeba_analyzed.meta.json`, together with a fingerprint of the expression list; the
steps that read the cache refuse one made with a different tagger or list, because its
words would disagree with how live text is analyzed. One cached record:

In [ ]:
with ANALYZED_PATH.open(encoding="utf-8") as f:
    records = (json.loads(line) for line in itertools.islice(f, 2050, None))
    record = next(r for r in records if r["en"])  # the first translated one

for key in ["id", "es", "en", "author"]:
    print(f"{key + ':':8}{record[key]}")
print("tokens: (surface form, [(lemma, pos), ...])")
for form, analyses in record["tokens"]:
    print(f"        {form!r:14} -> {analyses}")

### 2.2 From word-form counts to word frequencies

SUBTLEX-ESP counts forms; the word bank needs words. A form like *bajo* can belong to
several `(lemma, pos)` pairs, and the subtitles don't say which one was meant.

**Design decision: split each form's count by how that form is used in real
sentences.** For every form *f*, Tatoeba tells us how often spaCy assigned it each
analysis. The SUBTLEX count is divided in the same proportions:

$$\text{freq}(\ell, p) = \sum_{f} \text{count}_{\text{SUBTLEX}}(f) \cdot \frac{n_{\text{Tatoeba}}(f \to (\ell, p))}{n_{\text{Tatoeba}}(f)}$$

A contraction counts fully toward both of its words (*del* is always both *de* and *el*).

In [ ]:
from spanish_tutor.ingest.build_lexicon import form_analysis_counts, lemma_frequencies

occurrences, with_analysis = form_analysis_counts(read_analyzed(ANALYZED_PATH))

rows = []
for form in ["bajo", "como", "del"]:
    for (lemma, pos), n in with_analysis[form].most_common(4):
        rows.append((form, form_counts[form], f"{lemma}|{pos}", n, n / occurrences[form]))
pd.DataFrame(
    rows, columns=["form", "SUBTLEX count", "analysis", "Tatoeba tokens", "share"]
).style.format({"share": "{:.0%}", "SUBTLEX count": "{:,}"})

In [ ]:
frequencies, dropped = lemma_frequencies(form_counts, occurrences, with_analysis)
print(f"{len(frequencies):,} distinct (lemma, pos) pairs")
print(f"{dropped / total:.1%} of subtitle tokens are forms never seen in Tatoeba (dropped)\n")
pd.DataFrame(
    [
        (rank, lemma, pos, round(freq))
        for rank, ((lemma, pos), freq) in enumerate(frequencies.most_common(15), 1)
    ],
    columns=["rank", "lemma", "pos", "estimated count"],
)

**Design decision: keep only pairs that Wiktionary knows.** This filter removes names,
English words that appear in subtitles, and tagging errors the corrector couldn't
resolve. Lemma correction cut the rejects among the 3,000 most frequent pairs from 328
to about 20. What remains is ambiguous function words (*tal*, *solo*, *siquiera* tagged
as nouns) and abbreviations (*dr*, *srta*, *km*):

In [ ]:
top_pairs = frequencies.most_common(3000)
rejected = [(f"{l}|{p}", round(c)) for (l, p), c in top_pairs if (l, p) not in wiktionary]
print(
    f"of the top {len(top_pairs):,} pairs, {len(rejected)} have no Wiktionary entry. The most frequent:"
)
pd.DataFrame(rejected[:12], columns=["pair", "estimated count"]).T

### 2.3 The general lexicon

`python -m spanish_tutor.ingest.build_lexicon` turns all of the above into the `lexemes`
table: every `(lemma, pos)` in the lemmatized Tatoeba corpus that Wiktionary knows.

Design decisions:

- **Built in advance, independent of any learner.** Anyone who runs the pipeline gets the
  same standard lexicon. Any word the tutor meets already has an entry, so teaching it
  only means logging an event.
- **Each entry gets** a frequency per million subtitle words, an English definition, and an
  example sentence with its Tatoeba author.
- **Learner-independent examples.** The example is the most readable sentence for
  someone who knows the 2,000 most frequent words (about the size of a learner's
  dictionary's defining vocabulary). When a word is taught, a new example is chosen
  against the learner's own word bank.
- **Rebuilds never delete a referenced row.** `sql/fill_lexicon.sql` keeps every row
  that learner events (or reviews) point at, so they keep pointing at the right words.
  Frequencies are refreshed; definitions and examples only fill blanks, so reviewed fixes
  are never overwritten.
- **Rows a rebuild no longer produces** are left over from earlier builds that a later
  pipeline fix corrected away (*tambien* is now respelled *también*). If nothing
  references them they are deleted; otherwise they stay and only their frequency is
  cleared, because it described an older corpus. The first rebuild with this rule
  (2026-10-02) removed 8 rows and cleared 1 frequency (*mas* ADV, which has seed events).
- **Room for review.** `lexeme_reviews` will hold accuracy checks by a bilingual LLM or a
  person, append-only, without touching the sourced data.

Section 4.1 covers the database structure and example SQL queries in detail.

In [ ]:
conn = db.connect(DB_PATH)
pd.read_sql(
    """
    SELECT COUNT(*)                           AS words,
           COUNT(frequency_per_million)       AS with_frequency,
           COUNT(definition_en)               AS with_definition,
           COUNT(example_es)                  AS with_example,
           ROUND(100.0 * COUNT(example_es) / COUNT(*), 1) AS pct_with_example
    FROM lexemes
    """,
    conn,
)

In [ ]:
# Samples across the frequency range: very common, mid-frequency, rare.
pd.read_sql(
    """
    WITH ranked AS (
        SELECT lemma, pos, ROUND(frequency_per_million, 1) AS per_million,
               definition_en, example_es,
               ROW_NUMBER() OVER (ORDER BY frequency_per_million DESC) AS frequency_rank
        FROM lexemes
        WHERE frequency_per_million IS NOT NULL
    )
    SELECT * FROM ranked WHERE frequency_rank IN (1, 2, 3, 500, 501, 5000, 5001, 15000)
    """,
    conn,
)

### 2.4 Storing what you know: the word bank

The 1,500 most frequent lexicon words went into a spreadsheet, and each one you knew was
marked `r` (recognize) or `p` (can produce). `python -m spanish_tutor.seed build` recorded
the marks as events. The rest of the schema (`sql/schema.sql`) holds only learner data:

- **`word_events`**: an append-only log of every time a word is taught, seen, looked up or
  used. **Design decision:** the log is the source of truth, so scores can always be
  recalculated from full history.
- **`word_bank`**: one row per `(word, mode)`. Recognition and production are tracked
  separately, because the gap between understanding Spanish and producing it is what the
  project exists to close. **Design decision:** a trigger keeps this table current as
  events arrive. Computing it from the log on every read took 6.7 s per lookup at 1M
  events; the trigger table answers in 0.1 ms at 11M.

Schema changes reach existing databases through numbered files in `sql/migrations/`,
tracked in SQLite's `PRAGMA user_version`, so your data survives them. Section 4.1 shows
the constraints, the trigger in action, and example queries.

In [ ]:
print("schema version:", conn.execute("PRAGMA user_version").fetchone()[0])
pd.read_sql("SELECT mode, COUNT(*) AS words FROM word_bank GROUP BY mode", conn)

In [ ]:
# Words you recognize but can't yet produce, by part of speech: a ready-made practice list.
pd.read_sql(
    """
    SELECT l.pos, COUNT(*) AS words
    FROM word_bank AS r
    JOIN lexemes AS l ON l.lexeme_id = r.lexeme_id
    WHERE r.mode = 'recognition'
      AND NOT EXISTS (
          SELECT 1 FROM word_bank AS p
          WHERE p.lexeme_id = r.lexeme_id AND p.mode = 'production'
      )
    GROUP BY l.pos
    ORDER BY words DESC
    """,
    conn,
)

## 3. Vectorization

To find example sentences by **meaning** rather than by exact words, each sentence is
turned into a vector: a list of numbers positioned so that sentences with similar
meanings point in similar directions. Closeness is measured with **cosine similarity**
(1 = same direction, 0 = unrelated).

### 3.1 The model: `jina-embeddings-v2-base-es`

- **What:** a sentence-embedding model from Jina AI, trained on Spanish and English
  together. An English query finds Spanish sentences, and vice versa.
  Model card: [huggingface.co/jinaai/jina-embeddings-v2-base-es](https://huggingface.co/jinaai/jina-embeddings-v2-base-es).
- **Architecture:** JinaBERT, a 12-layer BERT variant that uses ALiBi attention biases
  instead of learned positions, which lets it read up to 8,192 tokens. Token vectors are
  mean-pooled into one 768-dimensional sentence vector.
- **License:** Apache-2.0.

Design decisions:

- **Why this model:** in a direct comparison with `multilingual-e5-base`, both ranked
  correctly, but jina separated related from unrelated sentences far more clearly:

  | Pair | jina | e5 |
  |---|---|---|
  | "Hace mucho calor hoy" ↔ "It is very hot today" | 0.895 | 0.924 |
  | "Hace mucho calor hoy" ↔ "Me gusta leer libros" | 0.038 | 0.841 |

  With jina, a similarity score means something on its own, so it can be used as a
  threshold.
- **The cost:** its architecture code is downloaded from Hugging Face
  (`trust_remote_code`), and that code doesn't run on transformers 5. So
  `transformers<5` and `sentence-transformers<6` are pinned, and the model and its code are
  pinned to exact commits (`src/spanish_tutor/embeddings.py`).
- **Normalized vectors:** every vector has length 1, so cosine similarity is simply the
  dot product.
- **CPU, via PyTorch:** about 130 sentences per second, so all 261k took about 35 minutes.

Loading the model (through LangChain's `HuggingFaceEmbeddings`) and printing its
structure confirms the pooling and the vector size:

In [ ]:
from spanish_tutor.embeddings import MODEL_ID, MODEL_REVISION, load_embeddings

embeddings = load_embeddings()
print(MODEL_ID, "@", MODEL_REVISION[:10])
print(embeddings._client)  # the underlying sentence-transformers model

### 3.2 One sentence and its vector

In [ ]:
anchor = "Tengo hambre, voy a comer algo."
vector = np.array(embeddings.embed_query(anchor))

print(f"sentence: {anchor!r}")
print(f"shape:    {vector.shape}   length (L2 norm): {np.linalg.norm(vector):.4f}")
print(f"range:    {vector.min():+.3f} … {vector.max():+.3f}\n")
pd.DataFrame(
    [vector[:10].round(4)], columns=[f"d{i}" for i in range(10)], index=["first 10 of 768"]
)

The individual numbers have no human meaning; only the *relative positions* of vectors
matter. To check that, we compare the anchor sentence with sentences that mean something
similar (in Spanish and English) and with sentences about unrelated topics.

### 3.3 Similar vs. different sentences

In [ ]:
examples = [
    # (sentence, group, language)
    (anchor, "anchor", "es"),
    ("Tengo mucha hambre.", "similar", "es"),
    ("¿Hay algo de comer en la cocina?", "similar", "es"),
    ("Quiero comer ahora mismo.", "similar", "es"),
    ("Vamos a almorzar juntos.", "similar", "es"),
    ("I'm hungry, I'm going to eat something.", "similar", "en"),
    ("I want to eat right now.", "similar", "en"),
    ("El tren sale a las ocho.", "different", "es"),
    ("Me gusta leer libros de historia.", "different", "es"),
    ("Mañana va a llover mucho.", "different", "es"),
    ("Mi hermano trabaja en un banco.", "different", "es"),
    ("The train leaves at eight.", "different", "en"),
]
frame = pd.DataFrame(examples, columns=["sentence", "group", "language"])
vectors = np.array(embeddings.embed_documents(frame["sentence"].tolist()))
frame["similarity to anchor"] = vectors @ vectors[0]  # unit vectors: dot product = cosine
frame.sort_values("similarity to anchor", ascending=False).style.format(
    {"similarity to anchor": "{:.3f}"}
)

### 3.4 Projecting to 2D with PCA

The vectors live in 768 dimensions. Principal component analysis finds the two directions
along which these twelve vectors vary most, and projects each vector onto them. If the
model works, the sentences about eating should land near the anchor, and the unrelated
sentences should land elsewhere.

**Caveat:** two dimensions keep only part of the variation (printed below), so distances
in the plot are approximate. Points that nearly coincide, like a sentence and its
translation, get their labels moved aside with thin leader lines. The cosine similarities above are the exact measure; the
plot is a visual check.

In [ ]:
pca = PCA(n_components=2)
points = pca.fit_transform(vectors)
frame["pc1"], frame["pc2"] = points[:, 0], points[:, 1]
explained = pca.explained_variance_ratio_
print(
    f"PC1 explains {explained[0]:.0%} and PC2 {explained[1]:.0%} of the variance ({explained.sum():.0%} together)"
)

# Palette: reference categorical slots 1 (blue) and 2 (orange), validated all-pairs;
# the anchor uses ink. Marker shape encodes language, so color is never the only cue.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
GROUP_STYLE = {
    "similar": {"color": "#2a78d6", "label": "Similar meaning"},
    "different": {"color": "#eb6834", "label": "Different topic"},
}
MARKER = {"es": "o", "en": "s"}

fig, ax = plt.subplots(figsize=(10, 6.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for group, style in GROUP_STYLE.items():
    for language, marker in MARKER.items():
        rows = frame[(frame.group == group) & (frame.language == language)]
        ax.scatter(
            rows.pc1,
            rows.pc2,
            s=90,
            marker=marker,
            color=style["color"],
            edgecolor=SURFACE,
            linewidth=2,
            zorder=3,
        )
a = frame.iloc[0]
ax.scatter(a.pc1, a.pc2, s=260, marker="*", color=INK, edgecolor=SURFACE, linewidth=2, zorder=4)

# Legend: color = meaning group, shape = language, star = anchor.
from matplotlib.lines import Line2D

handles = [
    Line2D([], [], marker="*", linestyle="", markersize=14, color=INK, label="Anchor sentence")
]
handles += [
    Line2D([], [], marker="o", linestyle="", markersize=9, color=s["color"], label=s["label"])
    for s in GROUP_STYLE.values()
]
handles += [
    Line2D([], [], marker=m, linestyle="", markersize=8, color=MUTED, label=lang)
    for lang, m in [("Spanish", "o"), ("English", "s")]
]
ax.legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
    fontsize=9,
    labelcolor=INK_2,
)

ax.set_xlabel(f"PC1 ({explained[0]:.0%} of variance)", color=MUTED, fontsize=9)
ax.set_ylabel(f"PC2 ({explained[1]:.0%} of variance)", color=MUTED, fontsize=9)
ax.set_title(
    "Sentences about eating cluster around the anchor, in both languages",
    loc="left",
    color=INK,
    fontsize=12,
    pad=12,
)
ax.grid(color=GRID, linewidth=0.6, zorder=0)
ax.tick_params(colors=MUTED, labelsize=8, length=0)
for side in ax.spines.values():
    side.set_visible(False)
# Extra room on the left: the anchor cluster sits near the left edge and needs label space.
span_x, span_y = np.ptp(points[:, 0]), np.ptp(points[:, 1])
ax.set_xlim(points[:, 0].min() - 0.55 * span_x, points[:, 0].max() + 0.4 * span_x)
ax.set_ylim(points[:, 1].min() - 0.2 * span_y, points[:, 1].max() + 0.2 * span_y)
plt.tight_layout()

# Label placement. Close points (a sentence and its translation) would stack their labels,
# so each label tries several positions and keeps the first that overlaps nothing already
# drawn: other labels, the markers, or the plot edge. Far positions get a thin leader line.
from matplotlib.text import Text
from matplotlib.transforms import Bbox

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
plot_area = ax.get_window_extent(renderer)
occupied = []
for x, y in zip(frame.pc1, frame.pc2):
    px, py = ax.transData.transform((x, y))
    occupied.append(Bbox.from_extents(px - 8, py - 8, px + 8, py + 8))

CANDIDATES = [(8, 3), (8, -13), (-8, 3), (-8, -13)]
CANDIDATES += [
    (dx, dy) for dist in (22, 38, 54, 70, 86, 102) for dy in (dist, -dist - 10) for dx in (16, -16)
]

for _, row in frame.iterrows():  # the anchor is first, so it gets the closest spot
    text = row.sentence if len(row.sentence) <= 34 else row.sentence[:32] + "…"
    style = {
        "fontsize": 9,
        "color": INK if row.group == "anchor" else INK_2,
        "fontweight": "bold" if row.group == "anchor" else "normal",
        "textcoords": "offset points",
        "zorder": 5,
    }
    for dx, dy in CANDIDATES:
        leader = (
            {"arrowstyle": "-", "color": MUTED, "lw": 0.6, "shrinkB": 5} if abs(dy) > 15 else None
        )
        label = ax.annotate(
            text,
            (row.pc1, row.pc2),
            xytext=(dx, dy),
            ha="left" if dx > 0 else "right",
            va="bottom",
            arrowprops=leader,
            **style,
        )
        # Measure the text alone: the leader line runs back to the point and would always
        # "collide" with the point's own marker.
        label.update_positions(renderer)
        box = Text.get_window_extent(label, renderer).expanded(1.02, 1.25)
        inside = plot_area.contains(box.x0, box.y0) and plot_area.contains(box.x1, box.y1)
        if inside and not any(box.overlaps(o) for o in occupied):
            occupied.append(box)
            break
        label.remove()
    else:  # nothing free: fall back to the default position
        ax.annotate(text, (row.pc1, row.pc2), xytext=CANDIDATES[0], va="bottom", **style)

plt.show()

In [ ]:
summary = (
    frame[frame.group != "anchor"]
    .groupby("group")["similarity to anchor"]
    .agg(["mean", "min", "max"])
)
summary.style.format("{:.3f}")

**Reading the result:** the similar sentences sit close to the anchor in the plot, and
the English translation lands next to its Spanish original, which is the cross-lingual
behavior the tutor relies on. The table confirms it without the 2D approximation: every
similar sentence scores higher than every different one.

## 4. Storage

The pipeline stores data in two places, each suited to its job:

| Store | Holds | Why this kind of store |
|---|---|---|
| **SQLite** (`data/processed/word_bank.db`) | The lexicon and everything about the learner | Structured, relational data with rules to enforce: keys, constraints, joins, transactions |
| **Chroma** (`data/processed/chroma/`) | Sentence vectors with metadata | Nearest-neighbor search over 768-dimensional vectors, which SQL databases don't do natively |

### 4.1 The SQL database

**Design decision: real SQL, not an ORM.** The schema (`sql/schema.sql`) and the
important queries are hand-written SQL, so every line can be read and defended on its
own. The SQL stays close to the ANSI standard, so it carries over to Databricks in
Phase 5.

#### Structure

```
 GENERAL DATA (same for everyone, rebuildable)  │  LEARNER DATA (personal)
                                                │
 lexemes ─────────────┬──< lexeme_reviews       │
 one row per          │    append-only checks   │
 (lemma, pos)         │    of lexicon accuracy  │
                      │                         │
                      └─────────────────────────┼──< word_events ──(trigger)──> word_bank
                                                │    append-only log           one row per
                                                │    of every encounter        (word, mode)
                                                │
                                                │    word_bank_rebuild (view): the same
                                                │    rows, recomputed from the full log
```

| Object | Kind | One row per | Written by |
|---|---|---|---|
| `lexemes` | table | word: `(lemma, pos)`, unique | `build_lexicon` (additive upsert) |
| `lexeme_reviews` | table | one review of one field of one word | the planned LLM/human review |
| `word_events` | table | one learning event (taught, seen, looked up, used) | the skills and the seed |
| `word_bank` | table, derived | `(word, mode)` the learner has acquired | **only** the trigger |
| `trg_word_bank_refresh` | trigger | | fires on every new `word_events` row |
| `word_bank_rebuild` | view | same rows as `word_bank`, from full history | used after formula changes |

The live database contains exactly these objects. This cell lists them straight from
SQLite's own catalog, `sqlite_master`, using a **read-only** connection so nothing in
this section can change your data:

In [ ]:
import sqlite3

ro = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)  # read-only
objects = pd.read_sql(
    """
    SELECT type, name, tbl_name AS on_table
    FROM sqlite_master
    WHERE name NOT LIKE 'sqlite_%'
    ORDER BY CASE type WHEN 'table' THEN 1 WHEN 'view' THEN 2 WHEN 'trigger' THEN 3 ELSE 4 END, name
    """,
    ro,
)
objects["rows"] = [
    ro.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] if kind in ("table", "view") else ""
    for kind, name in zip(objects["type"], objects["name"])
]
objects

`PRAGMA table_info` describes a table's columns: their types, whether they allow NULL,
and which form the primary key. The lexicon:

In [ ]:
pd.read_sql(
    "SELECT name, type, \"notnull\" AS not_null, pk AS primary_key FROM pragma_table_info('lexemes')",
    ro,
)

The rules live in the `CREATE TABLE` statements themselves. `word_events` shows the main
kinds: a foreign key (`REFERENCES`), `CHECK` constraints on allowed values, and a
table-level `CHECK` that ties each event type to its mode:

In [ ]:
print(ro.execute("SELECT sql FROM sqlite_master WHERE name = 'word_events'").fetchone()[0])

#### Constraints keep bad data out

Constraints make the database reject bad data itself, rather than trusting every piece
of code that writes to it. The demonstrations below run on a **throwaway in-memory
database** with the same schema, so your real one is never touched:

In [ ]:
demo = db.connect(":memory:")  # same schema, empty, discarded afterwards
db.init_schema(demo)
demo.execute("INSERT INTO lexemes (lemma, pos) VALUES ('estación', 'NOUN')")

attempts = {
    "a name as vocabulary (POS 'PROPN')": "INSERT INTO lexemes (lemma, pos) VALUES ('María', 'PROPN')",
    "the same word twice": "INSERT INTO lexemes (lemma, pos) VALUES ('estación', 'NOUN')",
    "an event for a word that doesn't exist": "INSERT INTO word_events (lexeme_id, mode, event_type, source) VALUES (999, 'recognition', 'taught', 'seed')",
    "'taught' logged as production": "INSERT INTO word_events (lexeme_id, mode, event_type, source) VALUES (1, 'production', 'taught', 'seed')",
    "a grade outside 0-5": "INSERT INTO word_events (lexeme_id, mode, event_type, source, grade) VALUES (1, 'production', 'used', 'conversation', 9)",
}
results = []
for attempt, sql in attempts.items():
    try:
        demo.execute(sql)
        results.append((attempt, "accepted"))
    except sqlite3.IntegrityError as error:
        results.append((attempt, f"rejected: {error}"))
pd.DataFrame(results, columns=["attempted insert", "result"])

#### The trigger: events in, word bank out

Nothing writes to `word_bank` directly. Each new event fires `trg_word_bank_refresh`,
which recomputes that one word's row from its own events (an indexed lookup, not a scan
of the whole log). Follow one word through its first few events:

In [ ]:
def log(event_type, grade=None, source="conversation"):
    mode = "production" if event_type == "used" else "recognition"
    demo.execute(
        "INSERT INTO word_events (lexeme_id, mode, event_type, source, grade) VALUES (1, ?, ?, ?, ?)",
        (mode, event_type, source, grade),
    )


def snapshot(step):
    rows = demo.execute(
        "SELECT mode, learned_via, familiarity, encounters FROM word_bank ORDER BY mode"
    ).fetchall()
    return [(step, *row) for row in rows] or [(step, "(no row)", "", "", "")]


story = [
    ("1. seen in a sentence, never taught", {"event_type": "seen"}),
    ("2. pre-taught", {"event_type": "taught", "source": "pre_teach"}),
    ("3. recognized easily (grade 5)", {"event_type": "seen", "grade": 5}),
    ("4. used it yourself (grade 3)", {"event_type": "used", "grade": 3}),
    ("5. had to look it up (grade 1)", {"event_type": "looked_up", "grade": 1}),
]
steps = []
for step, event in story:
    log(**event)
    steps += snapshot(step)
pd.DataFrame(steps, columns=["after", "mode", "learned_via", "familiarity", "encounters"])

Note step 1: seeing a word you were never taught doesn't add it to your word bank. And
familiarity is tracked separately per mode: a production grade never moves the
recognition score. (Familiarity is a placeholder, the mean of the last 5 grades scaled to
0–1, until Phase 6 replaces it with the SM-2 spaced-repetition algorithm.)

**Design decision: the log is the source of truth.** `word_bank_rebuild` recomputes every
row from the full history with window functions. It's too slow to read on every turn, but
it lets the scoring formula change later without losing anything. The trigger and the
rebuild must always agree, and a test checks that they do:

In [ ]:
live = demo.execute("SELECT * FROM word_bank ORDER BY lexeme_id, mode").fetchall()
rebuilt = demo.execute("SELECT * FROM word_bank_rebuild ORDER BY lexeme_id, mode").fetchall()
print("trigger table == full rebuild:", [tuple(r) for r in live] == [tuple(r) for r in rebuilt])

`lexeme_reviews` follows the same append-only idea. Reviews are never overwritten, so
the current verdict for each field is simply the latest one, which a window function
picks out:

In [ ]:
demo.executemany(
    "INSERT INTO lexeme_reviews (lexeme_id, field, verdict, suggestion, reviewer, reviewed_at) VALUES (1, ?, ?, ?, ?, ?)",
    [
        ("definition_en", "incorrect", "season", "claude-haiku-4-5", "2026-10-01 10:00:00"),
        ("definition_en", "unsure", None, "claude-opus-5-5", "2026-10-02 10:00:00"),
        ("definition_en", "correct", None, "human", "2026-10-03 10:00:00"),
        ("example_en", "correct", None, "claude-opus-5-5", "2026-10-02 10:00:00"),
    ],
)
pd.read_sql(
    """
    WITH ranked AS (
        SELECT field, verdict, reviewer, reviewed_at,
               ROW_NUMBER() OVER (PARTITION BY lexeme_id, field ORDER BY reviewed_at DESC) AS recency
        FROM lexeme_reviews
    )
    SELECT field, verdict AS latest_verdict, reviewer, reviewed_at FROM ranked WHERE recency = 1
    """,
    demo,
)

#### Example queries on your database

These run read-only against your real word bank and the general lexicon. Each shows a
SQL technique the tutor relies on.

**1. Join: your strongest words.** Words you can produce, with their lexicon data, most
frequent first:

In [ ]:
pd.read_sql(
    """
    SELECT l.lemma, l.pos, ROUND(l.frequency_per_million) AS per_million, l.definition_en
    FROM word_bank AS b
    JOIN lexemes   AS l ON l.lexeme_id = b.lexeme_id
    WHERE b.mode = 'production'
    ORDER BY l.frequency_per_million DESC
    LIMIT 8
    """,
    ro,
)

**2. Anti-join: what to learn next.** The most frequent words that *aren't* in your word
bank. `NOT EXISTS` finds rows with no match in another table. This is the simplest form
of the Phase 3 recommender, which ranks content by how many such words it contains.
The frequencies come from film subtitles, so informal and colloquial words rank high,
as they do in real conversation.

In [ ]:
pd.read_sql(
    """
    SELECT l.lemma, l.pos, ROUND(l.frequency_per_million, 1) AS per_million, l.definition_en
    FROM lexemes AS l
    WHERE l.frequency_per_million IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 FROM word_bank AS b
          WHERE b.lexeme_id = l.lexeme_id AND b.mode = 'recognition'
      )
    ORDER BY l.frequency_per_million DESC
    LIMIT 10
    """,
    ro,
)

**3. CTE + window function + conditional aggregation: coverage by frequency band.** How
much of each band of the frequency list you know, for recognition and for production. A
CTE (`WITH`) ranks every word with `ROW_NUMBER()`, `CASE` assigns bands, and conditional
`SUM`s count your words per band. This is the shape of the DELE/SIELE progress metric
("B1 vocabulary: recognize 68%, produce 41%"), with frequency bands standing in until
words carry CEFR levels.

In [ ]:
pd.read_sql(
    """
    WITH ranked AS (
        SELECT lexeme_id,
               ROW_NUMBER() OVER (ORDER BY frequency_per_million DESC) AS frequency_rank
        FROM lexemes
        WHERE frequency_per_million IS NOT NULL
    ),
    banded AS (
        SELECT r.lexeme_id,
               CASE WHEN frequency_rank <= 100  THEN '1. top 100'
                    WHEN frequency_rank <= 500  THEN '2. 101-500'
                    WHEN frequency_rank <= 1000 THEN '3. 501-1,000'
                    WHEN frequency_rank <= 2000 THEN '4. 1,001-2,000'
                    WHEN frequency_rank <= 5000 THEN '5. 2,001-5,000'
                    ELSE '6. beyond 5,000' END AS band,
               EXISTS (SELECT 1 FROM word_bank b WHERE b.lexeme_id = r.lexeme_id AND b.mode = 'recognition') AS recognized,
               EXISTS (SELECT 1 FROM word_bank b WHERE b.lexeme_id = r.lexeme_id AND b.mode = 'production')  AS produced
        FROM ranked AS r
    )
    SELECT band,
           COUNT(*) AS words,
           ROUND(100.0 * SUM(recognized) / COUNT(*), 1) AS pct_recognized,
           ROUND(100.0 * SUM(produced)   / COUNT(*), 1) AS pct_produced
    FROM banded
    GROUP BY band
    ORDER BY band
    """,
    ro,
)

**4. Weighted aggregate: how much of everyday Spanish you can follow.** Weighting each
word by its frequency gives *text coverage*: the share of words in typical speech that
are in your word bank. Research on vocabulary and comprehension (e.g. Nation, 2006)
suggests that following a text comfortably takes roughly 95–98% coverage. This estimate
counts word families loosely and ignores names, so treat it as indicative.

In [ ]:
pd.read_sql(
    """
    SELECT ROUND(100.0 * SUM(CASE WHEN b.lexeme_id IS NOT NULL THEN l.frequency_per_million END)
                 / SUM(l.frequency_per_million), 1) AS pct_of_spoken_tokens_recognized
    FROM lexemes AS l
    LEFT JOIN word_bank AS b ON b.lexeme_id = l.lexeme_id AND b.mode = 'recognition'
    WHERE l.frequency_per_million IS NOT NULL
    """,
    ro,
)

**5. Window function over time: word bank growth.** Words added per day with a running
total, `SUM(...) OVER (ORDER BY day)`. Right now everything arrived on seed day, so there
is one row. The same query becomes the Phase 7 growth chart as you learn.

In [ ]:
pd.read_sql(
    """
    SELECT DATE(learned_at) AS day,
           mode,
           COUNT(*) AS words_added,
           SUM(COUNT(*)) OVER (PARTITION BY mode ORDER BY DATE(learned_at)) AS running_total
    FROM word_bank
    GROUP BY day, mode
    ORDER BY day, mode
    """,
    ro,
)

**6. `EXPLAIN QUERY PLAN`: proof that lookups use indexes.** SQLite reports how it will
run a query. `SEARCH ... USING INDEX` means it jumps straight to the rows; `SCAN` would
mean reading the whole table. This is why word-bank lookups stay at about 0.1 ms even
with millions of events.

In [ ]:
for label, sql in {
    "one word's word-bank rows": "SELECT * FROM word_bank WHERE lexeme_id = 42",
    "one word's events, newest first": "SELECT * FROM word_events WHERE lexeme_id = 42 AND mode = 'recognition' ORDER BY occurred_at DESC",
    "a word by its text": "SELECT lexeme_id FROM lexemes WHERE lemma = 'estación' AND pos = 'NOUN'",
}.items():
    plan = ro.execute("EXPLAIN QUERY PLAN " + sql).fetchall()
    print(f"{label}:\n    " + "\n    ".join(row[-1] for row in plan))

**7. The upsert pattern: loading the lexicon.** `build_lexicon` stages rows in a
temporary table, then merges them with a single `INSERT ... SELECT ... ON CONFLICT DO
UPDATE`. New words are inserted; for existing words, frequencies are refreshed while
definitions and examples only fill blanks. `CASE` keeps an example's four fields
together. Two cleanup statements follow, each an anti-join (`NOT EXISTS`) against the
staging table: a `DELETE` of rows the build no longer produces and nothing references,
then an `UPDATE` clearing the frequency of those that have history. Every table that
references `lexemes` is listed in the `DELETE`; if a new one is added and not listed, the
foreign key fails the statement and the whole build rolls back. The whole file:

In [ ]:
from spanish_tutor.config import SQL_DIR

print((SQL_DIR / "fill_lexicon.sql").read_text(encoding="utf-8"))

**8. Migrations: changing the schema without losing data.** `schema.sql` always
describes the newest schema, for new databases. Existing databases are upgraded by the
numbered files in `sql/migrations/`, and SQLite's `PRAGMA user_version` records how many
have been applied.

- **Migration 1** added `frequency_per_million` to the lexicon.
- **Migration 2** added the `sessions` and `turns` tables, which record each conversation
  and its per-turn metrics (new words, retries, tokens, latency), and
  `word_events.turn_id`, which links every word event to the turn that caused it.
- **Migration 3** added `turns.kind`, which marks "¿cómo se dice?" translation turns so
  the adherence metric can leave them out, and renamed `correction_en` to `note_en`.
- **Migration 4** added `sessions.ended_at`, which tells an ended conversation from one
  left open, and the `session_summaries` table: the tutor's end-of-conversation notes,
  zero or one row per session. The summary's numbers aren't stored;
  `sql/queries/session_stats.sql` computes them from the log, including the words used
  for the first time ever (a `ROW_NUMBER()` over each word's `used` events).

A test keeps migrations honest: it loads a snapshot of the version-1 schema
(`tests/fixtures/schema_v1.sql`), upgrades it, and checks that every table matches a
new database's, column by column.

In [ ]:
from spanish_tutor.db import migrations

print("this database is at schema version", ro.execute("PRAGMA user_version").fetchone()[0])
for number, path in migrations():
    print(f"\n-- migration {number}: {path.name}\n" + path.read_text(encoding="utf-8"))
demo.close()

### 4.2 The Chroma vector store

[Chroma](https://docs.trychroma.com/) is an open-source vector database that runs locally
(stored in `data/processed/chroma/`, about 1.1 GB). `python -m spanish_tutor.ingest.index_tatoeba`
embedded all 261,405 translated Tatoeba sentences into one collection, `tatoeba`.

Design decisions:

- **One document per translated sentence.** The embedded text is the Spanish sentence.
  Untranslated sentences are skipped, because the learner needs the meaning.
- **Deterministic ids** (`tatoeba:<sentence id>`). Indexing skips ids already stored, so an
  interrupted run resumes and a re-run adds nothing.
- **Metadata travels with each vector:** the English translation, the author
  (attribution), the token count, and the sentence's vocabulary (its `(lemma, pos)` set from
  `lexicon.py`).
- **The vocabulary is stored as a string** (`"comer|VERB;casa|NOUN"`), because Chroma
  metadata values must be scalars. `;` separates items because multi-word lemmas
  contain spaces.
- **Cosine distance** for the index.
- **Model guard:** the collection records which embedding model built it, and
  `open_store()` refuses to open it with a different model. Vectors from different models
  live in different spaces, and mixing them would give plausible-looking garbage.

In [ ]:
from spanish_tutor.vectorstore import decode_vocab, open_store, search_sentences

store = open_store()
collection = store._collection
print(f"collection {collection.name!r}: {collection.count():,} sentences")
print("collection metadata (model guard):", collection.metadata)

One stored record: its text, metadata and vector. Re-embedding the same text gives the
same vector (similarity 1.000), which confirms the store holds exactly what the model
produces:

In [ ]:
stored = collection.get(ids=["tatoeba:2503"], include=["documents", "metadatas", "embeddings"])
if not stored["ids"]:  # fall back to any stored record
    stored = collection.get(limit=1, include=["documents", "metadatas", "embeddings"])

text, meta, stored_vector = (
    stored["documents"][0],
    stored["metadatas"][0],
    np.array(stored["embeddings"][0]),
)
print("id:      ", stored["ids"][0])
print("text:    ", text)
for key, value in meta.items():
    print(f"{key + ':':9} {value}")
print("vocab →  ", sorted(decode_vocab(meta["vocab"])))
print(f"\nvector: {stored_vector.shape}, first 5: {stored_vector[:5].round(4)}")
print(
    f"similarity to a fresh embedding of the same text: {stored_vector @ np.array(embeddings.embed_query(text)):.3f}"
)

## 5. Lookups

### 5.1 Vocabulary-constrained sentence search

This is the core retrieval step the conversation skill uses. `search_sentences()`:

1. Embeds the query and fetches the 50 most similar sentences from Chroma.
2. Compares each sentence's vocabulary with your word bank (recognition mode).
3. Keeps sentences with at most `max_unknown` words you don't know: 0 means fully readable,
   1 means one new word to teach (Krashen's *i+1*).
4. Returns up to `k` in similarity order, listing any new words.

**Design decision:** the vocabulary filter runs in Python after the similarity search.
That's simple and fast enough for Phase 1. Phase 3's SQL difficulty index is its
long-term home.

First, the words you know, read from SQLite:

In [ ]:
known = db.known_vocabulary(
    conn
)  # SELECT lemma, pos FROM word_bank JOIN lexemes WHERE mode = 'recognition'
print(f"{len(known):,} words you recognize, e.g. {sorted(known)[:6]}")

Without the filter, plain similarity search returns sentences with words you haven't
learned. The `unknown words` column shows what the filter would remove:

In [ ]:
query = "I'm hungry"
raw_hits = store.similarity_search_with_score(query, k=10)
pd.DataFrame(
    [
        (
            1 - distance,
            doc.page_content,
            doc.metadata["en"],
            ", ".join(f"{l}|{p}" for l, p in sorted(decode_vocab(doc.metadata["vocab"]) - known)),
        )
        for doc, distance in raw_hits
    ],
    columns=["similarity", "Spanish", "English", "unknown words"],
).style.format({"similarity": "{:.3f}"})

In [ ]:
def show(hits):
    return pd.DataFrame(
        [
            (
                h.similarity,
                h.es,
                h.en,
                ", ".join(f"{l}|{p}" for l, p in sorted(h.unknown)),
                h.author,
            )
            for h in hits
        ],
        columns=["similarity", "Spanish", "English", "new words", "author"],
    ).style.format({"similarity": "{:.3f}"})


print(f"query {query!r}, only words you know:")
show(search_sentences(store, query, known, k=5, max_unknown=0))

In [ ]:
query = "¿Dónde está la estación?"
print(f"query {query!r}, one new word allowed (i+1):")
show(search_sentences(store, query, known, k=5, max_unknown=1))

### 5.2 Looking up words in the word bank

For each word, this query shows everything stored about it: its general lexicon data
(frequency, definition, attributed example) and whether *you* recognize and produce it.
*estación*, the new word from the search above, is in the lexicon but not in your word
bank: teaching it only needs a `taught` event. The query starts from the list of
requested words and `LEFT JOIN`s the tables, so even a word missing from the lexicon
would still appear, with empty fields:

In [ ]:
pd.read_sql(
    """
    WITH requested(lemma) AS (VALUES ('hambre'), ('comer'), ('estación'))
    SELECT r.lemma, l.pos, ROUND(l.frequency_per_million, 1) AS per_million,
           l.definition_en, l.example_es, l.example_en,
           l.example_source, l.example_author,
           MAX(CASE WHEN b.mode = 'recognition' THEN 'yes' ELSE 'no' END) AS recognize,
           MAX(CASE WHEN b.mode = 'production'  THEN 'yes' ELSE 'no' END) AS produce,
           MIN(b.learned_at) AS learned_at
    FROM requested AS r
    LEFT JOIN lexemes   AS l ON l.lemma = r.lemma
    LEFT JOIN word_bank AS b ON b.lexeme_id = l.lexeme_id
    GROUP BY r.lemma, l.lexeme_id
    """,
    conn,
).set_index("lemma").T

### 5.3 Resolving a learner's own words

Text the learner types goes through the same `analyze()`, but it needs extra safeguards
before anything is logged. `words.LexiconIndex` maps each `(lemma, pos)` to a lexicon row:

1. An exact match.
2. **Missing accents restored** when exactly one word fits: a learner on an English
   keyboard writes *jardin* for *jardín*. Of 26k lexicon words, only about 130 become
   ambiguous without accents, almost all function-word pairs (*si*/*sí*, *el*/*él*), and
   those are never guessed. A spelling that is a word in its own right (*si*, *esta*) is never
   re-accented.
3. A real word missing from the lexicon is added from Wiktionary.
4. **Anything else is not a word and is never logged.** spaCy turns the misspelling *sabo*
   (for *sé*) into a verb `sabo`; without this check it would enter the word bank as
   vocabulary.

A word typed on its own gets no help from context: spaCy tags a lone *perro* as a name.
`headword()` matches dictionary forms directly instead, taking the most frequent part of
speech.

Misspellings that Wiktionary lists as entries (*dia*, *tambien*) never reach this step:
the lemma corrector already respelled them (section 2.1).

The demo builds the index without Wiktionary, so it can't add rows to your database.

In [ ]:
from spanish_tutor.words import LexiconIndex

index = LexiconIndex(conn)  # no Wiktionary: read-only
cases = [
    (("casa", "NOUN"), "exact match"),
    (("jardin", "NOUN"), "missing accent"),
    (("si", "SCONJ"), "a word in its own right: never sí"),
    (("sabo", "VERB"), "misspelling (for sé)"),
]
rows = []
for analysis, case in cases:
    found = index.resolve(analysis)
    result = f"{found.lemma}|{found.pos}" if found else "(not a word: never logged)"
    rows.append(("|".join(analysis), case, result))
for word in ["perro", "bajo", "jardin"]:
    found = index.headword(word)
    rows.append((f"{word} (typed alone)", "headword lookup", f"{found.lemma}|{found.pos}"))
pd.DataFrame(rows, columns=["input", "case", "resolves to"])

### 5.4 A readable example for a new word

Each lexicon word has one stored example, chosen for a generic learner who knows the
2,000 most frequent words. Measured against your word bank, it is fully readable only
about 61% of the time. So when a word is taught, `vectorstore.find_example()` looks for a
better one: the sentence closest in meaning to the word's definition that contains the
word and *no other* word you don't know. It finds one for about 72% of the words whose
stored example fails, so about 89% of taught words come with an example you can read in
full.

Some of the most frequent words you don't know yet, whose stored example contains a
second unknown word:

In [ ]:
from spanish_tutor.vectorstore import find_example

candidates = pd.read_sql(
    """
    SELECT l.lemma, l.pos, l.definition_en, l.example_es
    FROM lexemes AS l
    WHERE l.frequency_per_million IS NOT NULL
      AND l.example_es IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM word_bank AS b
                      WHERE b.lexeme_id = l.lexeme_id AND b.mode = 'recognition')
    ORDER BY l.frequency_per_million DESC
    LIMIT 300
    """,
    conn,
)
rows = []
for word, tokens in zip(candidates.itertuples(), analyze(candidates.example_es.tolist())):
    target = (word.lemma, word.pos)
    others = vocabulary(tokens) - known - {target}
    if not others:
        continue  # the stored example is already readable
    hit = find_example(store, target, known, f"{word.lemma}: {word.definition_en}")
    found = hit.es if hit else "(none: the stored example is shown with glosses)"
    rows.append((word.lemma, word.example_es, ", ".join(sorted(l for l, _ in others)), found))
    if len(rows) == 6:
        break
pd.DataFrame(
    rows, columns=["word", "stored example", "other unknown words", "readable example found"]
)

## 6. Content and the difficulty index (Phase 3)

The recommender ranks songs, stories and book chapters by how many words each would
teach. That needs every item's full vocabulary in the database, indexed once when the item
is added, so ranking is one SQL join against the word bank rather than a re-analysis of
the text.

Schema (migration 6, designed with Jason on 2026-10-05):

| Table | What it holds |
|---|---|
| `books` | A book's title, author, source, and whether it's copyrighted (local use only) |
| `content_items` | Songs, stories and chapters (a chapter points to its book and has a number), with the text and indexing status |
| `content_vocab` | **The difficulty index**: each item's lexemes with their occurrence counts |
| `content_events` | Append-only: what the learner started and finished, and whether it was the recommendation or their own request |
| `word_resolutions` | Append-only: how forms no dictionary knows were resolved, and the cache that keeps each form to one model call |

Everything below runs on an **in-memory copy** of your word bank, migrated to version 6.
Your real database is never written.

In [ ]:
from spanish_tutor import content, progress, recommend
from spanish_tutor.ingest.expressions import definitions
from spanish_tutor.ingest.gutenberg import GUTENBERG_DIR, body, load_manifest, split_chapters
from spanish_tutor.lexicon import analyze, load_wiktionary
from spanish_tutor.words import LexiconIndex

sandbox = db.connect(":memory:")
conn.backup(sandbox)  # a copy of your word bank
db.init_schema(sandbox)  # migration 6, on the copy only
print("copy at schema version", sandbox.execute("PRAGMA user_version").fetchone()[0])

### 6.1 A public-domain book, split for analysis

The test book is Horacio Quiroga's *Cuentos de amor de locura y de muerte* (1917,
Project Gutenberg #13507): 18 stories, each a chapter. Run
`uv run python -m spanish_tutor.ingest.gutenberg 13507` once to download and split it.

Every Gutenberg book is laid out differently, so each has a small hand-written **manifest**
(`ingest/books/<ebook>.json`, committed; the text never is): the chapter headings as they
appear in the text with the titles to show, where the reading text ends (a vocabulary or
notes section), patterns to remove (margin line numbers, English notes, illustrations) and
substitutions (markup, dialogue dashes). Headings must match whole lines, in order, or the
split fails loudly instead of guessing.

Real text turned up three problems the tests hadn't, each fixed by a general rule:

1. **Hard-wrapped lines.** Gutenberg's text files break lines at about 70 characters. Split
   at every line break, the tagger saw half sentences (*y le jura no*), and tagged *no* as a
   particle. Prose is now split into paragraphs at blank lines, and a paragraph's own
   line breaks are joined. Songs still split by line.
2. **An ellipsis glued to an opening mark.** In *queja...¿no?*, spaCy keeps *...¿no* as one
   token. An opening *¿* or *¡* always starts a new sentence or clause, so the text is split
   there.
3. **Spellings from before the 1952 reform.** *fué*, *dió* and *fuí* are entries in
   Wiktionary (marked "superseded spelling … deprecated in 1952"). So they were added as
   words of their own: *fué* appeared 74 times. They are now respelled in the text before
   tagging, next to accent restoration, so *fué* is analyzed exactly like *fue*. The
   2010-reform spellings (*sólo*, *guión*) are left alone: they're still common in modern
   text. Older books (1909-1919) also wrote the one-letter words *á*, *é*, *ú* and accents
   such as *ántes* and *órden*, which Wiktionary lists as "obsolete spelling of …": the
   preposition *a* became a separate, frequent "unknown" word. The 38 obsolete spellings
   that differ from the modern word **only in accent marks** are respelled too (measured
   2026-10-06: 2.4 points of a 1909 reader's unknown share). Obsolete spellings with other
   changes (*ay* → *hay*) are left alone.

   Measured while fixing this, and not fixed: the tagger lemmatizes *fue* and *fueron*
   as *ser* even when they mean "went". All 7,098 Tatoeba tokens are *ser*.
   *fui* and *fuimos* always become *ir*.

In [ ]:
book = split_chapters(
    body((GUTENBERG_DIR / "pg13507.txt").read_text(encoding="utf-8")), load_manifest(13507)
)
print(f"{len(book)} chapters, {sum(len(t.split()) for _, t in book):,} words")

wrapped = (
    "como ella que lo mejor era separarnos y le jura no\nquererla más. Se\nfue.\n\nUna queja...¿no?"
)
old_spelling = "Aquello fué lo mejor, y se lo dió á su padre ántes de irse."
pd.DataFrame(
    [
        ("hard-wrapped paragraph", wrapped, content.sentences(wrapped, "story")),
        ("pre-reform spellings", old_spelling, corrector.restorer.restore_text(old_spelling)),
    ],
    columns=["case", "text", "analyzed as"],
)

### 6.2 Indexing: every word resolved to a lexeme

`content.index_item` analyzes the text with the same `analyze()` as everything else, so the
index and the word bank agree on what a word is. Each `(lemma, pos)` is resolved in order,
and the first rule that applies wins:

1. A word already in the lexicon (with accents restored if needed).
2. A word Wiktionary knows, or an approved expression, which is added to `lexemes`.
3. An earlier decision for the same form, from `word_resolutions`.
4. Otherwise the form is **pending**. All of an item's pending forms go to Claude in one
   structured call (`resolve.py`, below). Without the model, they count as unresolved.

Tokens are counted per analysis, so *del* counts as *de* + *el*, and an expression counts
once. Names aren't counted.

Here five stories are indexed without the model: three as a small book, two as standalone
stories.

In [ ]:
lexicon_index = LexiconIndex(sandbox, load_wiktionary(), definitions())
fingerprint = content.analysis_fingerprint()


def analyze_many(units):
    return analyze(units, corrector=corrector)


with sandbox:
    book_id, chapter_ids = content.add_book(
        sandbox,
        "Cuentos (3 stories)",
        book[9:12],
        source="gutenberg:13507",
        is_private=False,
        author="Horacio Quiroga",
    )
    story_ids = [
        content.add_item(
            sandbox, "story", title, text, source="gutenberg:13507", is_private=False,
            author="Horacio Quiroga",
        )
        for title, text in (book[2], book[6])
    ]  # fmt: skip
indexed = []
for content_id in chapter_ids + story_ids:
    r = content.index_item(sandbox, lexicon_index, content_id, analyze_many, fingerprint)
    title = sandbox.execute(
        "SELECT title FROM content_items WHERE content_id = ?", (content_id,)
    ).fetchone()[0]
    indexed.append((title, r.tokens, r.words, r.unresolved_tokens, len(r.outcomes)))
pd.DataFrame(
    indexed, columns=["item", "running words", "distinct words", "unresolved", "pending forms"]
)

### 6.3 Forms no dictionary knows: Claude decides, the database checks

About 0.7% of the book's running words are in neither the lexicon nor Wiktionary. They
include old accents (*concluído*), old enclitic verbs (*sentóse*), Guaraní and regional
words (*tacuapí*, *chircal*), English (*fox-terrier*, *flirt*), and tagger slips. Claude
Opus 5.5 sees each form with the sentence it appears in and the tagger's guess, and gives
one of three verdicts:

- **variant**: another spelling of an existing word. It is counted as that word, so a
  known word isn't counted as new.
- **word**: a real word the dictionaries miss. It is added with Claude's definition,
  labeled *model-written* on lesson cards. Its example is the real sentence where it was
  met, with Claude's translation.
- **not_spanish**: excluded from the vocabulary.

The model only proposes. Every claim is checked before it is written: an invalid part of
speech, a variant of a word that exists nowhere, or a new word without a definition is
rejected. Every decision is stored, so each form is sent once, ever.

The notebook doesn't call the model, because that costs money. These are the decisions
stored by the real run over the whole book, read from `word_resolutions` in your database.
Answers the checks rejected aren't stored (22 in that run), so a re-index asks about them
again.

In [ ]:
review = pd.read_sql(
    """
    SELECT w.form, w.tagged_lemma || '|' || w.tagged_pos AS tagged, w.verdict,
           l.lemma, l.pos, w.reason
    FROM word_resolutions AS w
    LEFT JOIN lexemes AS l ON l.lexeme_id = w.lexeme_id
    ORDER BY w.resolution_id
    """,
    ro,
)
print(review.verdict.value_counts().to_string())
pd.concat([review[review.verdict == v].head(4) for v in ("variant", "word", "not_spanish")])

### 6.4 Ranking: two recommenders, both plain SQL

**Songs and stories** (`sql/queries/recommend_items.sql`) are ranked by the number of
distinct words each would teach, fewest first. That is the pre-teaching burden, because
every new word gets taught. Ties go to the item with more of its running words already
known. The query `LEFT JOIN`s each item's vocabulary to the recognition word bank and
counts the words that find no match.

**The ceiling.** Anything with more than 20% of its running words unknown
(`recommend.MAX_UNKNOWN_SHARE`; books across the whole book) is never suggested and is
listed apart as "too hard for now". The same query ranks either side (`:too_hard`). These
Quiroga stories are about 28% unknown, so they're on the too-hard side; the ranking and
the chapter order work the same there.

In [ ]:
print(progress.query("recommend_items"))
pd.concat(
    [
        pd.DataFrame(
            recommend.items(sandbox), columns=["title", "new_words", "tokens", "coverage"]
        ),
        pd.DataFrame(
            recommend.items(sandbox, too_hard=True),
            columns=["title", "new_words", "tokens", "coverage"],
        ),
    ],
    keys=["suggested", "too hard for now"],
)

**Books** (`sql/queries/recommend_books.sql`) are ranked by **new-word density across
the whole book**: unknown running words divided by all running words. Ranked by its first
chapter, a hard book with a short, easy preface would look easy (Jason's case, now a test).

Reading order is part of the query. A started book's next chapter is the first one after
the highest finished chapter, and it always comes before any new book is begun. Here,
starting and finishing chapter 1 moves the book to chapter 2:

In [ ]:
cols = [
    "title", "state", "density", "new_words", "next_chapter_no", "next_chapter_title",
    "next_chapter_new_words",
]  # fmt: skip
before = pd.DataFrame(recommend.books(sandbox, too_hard=True))[cols]
content.start(sandbox, chapter_ids[0], "recommended")
content.finish(sandbox, chapter_ids[0])
after = pd.DataFrame(recommend.books(sandbox, too_hard=True))[cols]
pd.concat([before, after], keys=["before reading", "after chapter 1"])

What pre-teaching the next chapter would take: its new words, most frequent in the
chapter first (`sql/queries/item_new_words.sql`):

In [ ]:
pd.DataFrame(recommend.new_words(sandbox, chapter_ids[1])).head(10)[
    ["lemma", "pos", "occurrences", "definition_en", "model_written"]
]

**Measured on the whole book (2026-10-05):**

- Indexing: about 47,000 counted running words in ~2 minutes on CPU, nearly all of it
  tagging.
- The ranking queries take 12 ms on the real book. On synthetic data at a scale far beyond
  personal use (6,000 items × 500 words, 3 million index rows), they take under a second.
- The book is far above the current word bank: 28% of its running words are unknown, and
  it has 4,284 distinct new words, against the ~98% coverage usually needed for
  comfortable reading. Classic literature ranks fine, but all of it is hard at this level.

**Easier public-domain texts (slice 4.1, measured 2026-10-06).** Nine Gutenberg candidates
(fables, children's stories, graded readers written for English-speaking students around
1910) were measured against the word bank, read-only, before anything was added:

| Book | Unknown share | If the bank also had every unreviewed word to rank 5,000 |
|---|---|---|
| An Elementary Spanish Reader (Harrison, 1912) | 19.4% (properly split) | 12.0% |
| Libro segundo de lectura (Cyr) | 23.2% | 15.0% |
| Lecturas fáciles con ejercicios | 28.5% | 16.9% |
| Spanish Tales for Beginners | 26.4% | 17.0% |
| Six others (Samaniego, Coloma, readers, short stories) | 29–33% | |

No real text came near the original 10% ceiling, even with a much larger word bank, so
Jason raised it to 20%. About 6% of these books' running words are frequent words (rank
≤ 1,500) not in the bank: a third are the 125 expressions not yet marked (*por qué*,
*después de*), the rest words left unmarked in the seed review. An Elementary Spanish
Reader (21 tales and fables) passes, and its first chapter needs 14 new words.

## Summary

| Stage | What happens | Code |
|---|---|---|
| Download | Three sources into `data/raw/`; code is committed, data never is | `ingest/download.py` |
| Parse | SUBTLEX's three-block sheet, Tatoeba's TSVs joined with authors, compact Wiktionary entries | `ingest/subtlex.py`, `ingest/tatoeba.py`, `ingest/wiktionary.py` |
| Lemmatize | spaCy's transformer pipeline (`es_dep_news_trf`) plus project rules turn text into `(lemma, pos)`; one shared path | `lexicon.py` |
| Correct | Lemmas that aren't words, or are the wrong word for the form, are repaired from Wiktionary's form records; lowercase words mis-tagged as names are re-tagged; missing accents are restored before tagging | `lexicon.LemmaCorrector`, `lexicon.AccentRestorer` |
| Frequency | Form counts split by real usage | `ingest/build_lexicon.py` |
| Expressions | Wiktionary's multi-word entries, rule-filtered and reviewed by Claude; matched by lemma sequence as one item | `ingest/expressions.py`, `lexicon.ExpressionMatcher` |
| Lexicon | Every corpus word Wiktionary knows, plus the approved expressions, with frequency, definition and example; learner-independent; rebuilds keep every referenced row | `ingest/build_lexicon.py`, `sql/fill_lexicon.sql` |
| Word bank | Event log plus trigger-maintained table in SQLite, recognition and production separate; every event linked to the turn that caused it | `sql/schema.sql`, `sql/migrations/`, `db.py`, `seed.py` |
| Vectorize | jina-embeddings-v2-base-es, 768-d unit vectors, CPU | `embeddings.py` |
| Store | Chroma collection with vocabulary metadata and a model guard | `vectorstore.py`, `ingest/index_tatoeba.py` |
| Look up | Similarity search filtered to words you know (or i+1); a readable example for each new word | `vectorstore.search_sentences`, `vectorstore.find_example` |
| Resolve | A learner's words to lexicon rows: accents restored, headwords matched, non-words rejected | `words.LexiconIndex` |
| Index content | Songs, stories and book chapters analyzed once; each item's words and counts stored; unknown forms resolved by Claude and checked | `content.py`, `resolve.py`, `ingest/gutenberg.py` |
| Recommend | Songs and stories by fewest new words; books by whole-book density, chapters in order | `recommend.py`, `sql/queries/recommend_*.sql` |

In [ ]:
conn.close()